In [1]:
import requests

url = "https://puckpedia.com/team/new-york-rangers"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

In [2]:
import asyncio
from datetime import datetime, timezone

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE_URL = "https://puckpedia.com"

TEAM = {
    "team_slug": "new-york-rangers",
    "team_name": "New York Rangers",
    "url": "https://puckpedia.com/team/new-york-rangers",
}

In [3]:

def money_to_int(value):
    """
    Convert:
        "$13,250,000"
        "13,250,000"
        "$0"
        None

    into an integer or None.
    """

    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    negative = value.startswith("-")

    value = (
        value
        .replace("$", "")
        .replace(",", "")
        .replace("-", "")
        .strip()
    )

    if not value:
        return None

    try:
        number = int(float(value))
    except ValueError:
        return None

    return -number if negative else number


async def fetch_team_html(page, team_name, url):
    """
    Load one PuckPedia team page and return the rendered HTML.
    """

    print(f"Loading {team_name}...")
    print(f"  {url}")

    response = await page.goto(
        url,
        wait_until="domcontentloaded",
        timeout=60000,
    )

    if response is not None:
        print(f"  HTTP status: {response.status}")

    # Wait for an actual contract salary cell.
    await page.wait_for_selector(
        "table.pp_table-roster td[data-sal]",
        timeout=60000,
    )

    html = await page.content()

    if "Just a moment..." in html:
        raise RuntimeError(
            f"Cloudflare challenge returned for {team_name}."
        )

    return html


In [4]:

def parse_contract_page(
    html,
    team_slug,
    team_name,
    source_url,
):
    """
    Parse every PuckPedia player contract table.

    Produces one row per:
        team
        player
        contract year

    Stores both:
        year   -> 1, 2, 3...
        season -> 2026-27, 2027-28...
    """

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    # -----------------------------------------------------------------
    # Find contract tables only.
    #
    # Excludes:
    # - salary-cap summary table
    # - GearGeek equipment tables
    # -----------------------------------------------------------------

    contract_tables = []

    for table in soup.select(
        "table.pp_table-roster"
    ):

        has_player = bool(
            table.select_one(
                'a[href^="/player/"]'
            )
        )

        has_salary = bool(
            table.select_one(
                "td[data-sal]"
            )
        )

        if has_player and has_salary:
            contract_tables.append(table)

    if not contract_tables:
        raise ValueError(
            f"No player contract tables found for {team_name}."
        )

    records = []

    scrape_datetime = datetime.now(
        timezone.utc
    )

    # -----------------------------------------------------------------
    # Parse every contract table
    # -----------------------------------------------------------------

    for table in contract_tables:

        # -------------------------------------------------------------
        # Identify which section this table belongs to
        # -------------------------------------------------------------

        section = "Unknown"

        wrapper = table.find_previous(
            "div",
            id=lambda x: x and x.startswith("capby_")
        )

        if wrapper:

            header = wrapper.find("div", class_="flex items-center")

            if header:

                divs = header.find_all("div", recursive=False)

                # div[0] = player count
                # div[1] = section name
                if len(divs) >= 2:
                    section = divs[1].get_text(strip=True)

        print(f"Section: {section}")

        # -------------------------------------------------------------
        # Read season headings for THIS table
        #
        # Example:
        # 2026-27
        # 2027-28
        # 2028-29
        # -------------------------------------------------------------

        season_headers = []

        for th in table.select("thead th"):

            text = th.get_text(
                " ",
                strip=True,
            )

            # Season headers on these tables have a data-column
            # greater than zero and text such as 2026-27.
            if (
                len(text) == 7
                and text[:4].isdigit()
                and text[4] == "-"
                and text[5:].isdigit()
            ):
                season_headers.append(text)

        # -------------------------------------------------------------
        # Player rows
        # -------------------------------------------------------------

        for row in table.select(
            "tbody > tr"
        ):

            player_link = row.select_one(
                'a[href^="/player/"]'
            )

            if player_link is None:
                continue

            # ---------------------------------------------------------
            # Player name
            # "Matthews, Auston" -> "Auston Matthews"
            # ---------------------------------------------------------

            raw_name = player_link.get_text(
                " ",
                strip=True,
            )

            if "," in raw_name:

                last_name, first_name = [
                    value.strip()
                    for value
                    in raw_name.split(",", 1)
                ]

                player = (
                    f"{first_name} {last_name}"
                )

            else:
                player = raw_name

            relative_player_url = (
                player_link.get("href", "")
            )

            player_url = (
                BASE_URL
                + relative_player_url
            )

            # ---------------------------------------------------------
            # Position / goalie catches
            # ---------------------------------------------------------

            position = None
            catches = None

            first_td = row.find("td")

            if first_td is not None:

                for detail in first_td.select(
                    "div.text-xs > div"
                ):

                    spans = detail.find_all(
                        "span"
                    )

                    if len(spans) < 2:
                        continue

                    label = (
                        spans[0]
                        .get_text(
                            " ",
                            strip=True,
                        )
                        .lower()
                    )

                    value = (
                        spans[-1]
                        .get_text(
                            " ",
                            strip=True,
                        )
                        .upper()
                    )

                    if label == "pos":
                        position = value

                    elif label == "catches":
                        position = "G"
                        catches = value

            # ---------------------------------------------------------
            # Contract salary cells
            # ---------------------------------------------------------

            salary_cells = row.select(
                "td[data-sal]"
            )

            for year, td in enumerate(
                salary_cells,
                start=1,
            ):

                # -----------------------------------------------------
                # Match contract year to actual season
                # -----------------------------------------------------

                if year <= len(
                    season_headers
                ):
                    season = (
                        season_headers[
                            year - 1
                        ]
                    )
                else:
                    season = None

                if season is None:
                    raise ValueError(
                        f"Could not identify season "
                        f"for {player}, year {year}, "
                        f"{team_name}."
                    )

                html_cell = str(td).lower()

                # -----------------------------------------------------
                # Contract flags
                # -----------------------------------------------------

                no_movement_clause = (
                    "no movement clause"
                    in html_cell
                )

                modified_no_trade_clause = (
                    "modified no trade clause"
                    in html_cell
                )

                no_trade_clause = (
                    "no trade clause"
                    in html_cell
                    and not
                    modified_no_trade_clause
                )

                two_way_contract = (
                    "two-way contract"
                    in html_cell
                    or
                    "two way contract"
                    in html_cell
                )

                performance_bonus = (
                    "performance bonus"
                    in html_cell
                )

                # -----------------------------------------------------
                # Record
                # -----------------------------------------------------

                records.append({
                    "team_slug":
                        team_slug,

                    "team_name":
                        team_name,

                    "player":
                        player,

                    "player_url":
                        player_url,

                    "position":
                        position,

                    "catches":
                        catches,

                    "year":
                        year,

                    "season":
                        season,

                    "cap_hit":
                        money_to_int(
                            td.get(
                                "data-ch"
                            )
                        ),

                    "aav":
                        money_to_int(
                            td.get(
                                "data-aav"
                            )
                        ),

                    "total_salary":
                        money_to_int(
                            td.get(
                                "data-sal"
                            )
                        ),

                    "signing_bonus":
                        money_to_int(
                            td.get(
                                "data-sb"
                            )
                        ),

                    "performance_bonus_amount":
                        money_to_int(
                            td.get(
                                "data-bonus"
                            )
                        ),

                    "no_movement_clause":
                        no_movement_clause,

                    "no_trade_clause":
                        no_trade_clause,

                    "modified_no_trade_clause":
                        modified_no_trade_clause,

                    "two_way_contract":
                        two_way_contract,

                    "performance_bonus":
                        performance_bonus,

                    "source_url":
                        source_url,

                    "scrape_datetime":
                        scrape_datetime,

                    "contract_section":
                        section,
                })

    if not records:
        raise ValueError(
            f"Contract tables were found for {team_name}, "
            "but no player records were extracted."
        )

    df = pd.DataFrame(records)

    # -----------------------------------------------------------------
    # Final column order
    # -----------------------------------------------------------------

    df = df[
        [
            "team_slug",
            "team_name",
            "player",
            "player_url",
            "position",
            "catches",
            "year",
            "season",
            "cap_hit",
            "aav",
            "total_salary",
            "signing_bonus",
            "performance_bonus_amount",
            "no_movement_clause",
            "no_trade_clause",
            "modified_no_trade_clause",
            "two_way_contract",
            "performance_bonus",
            "source_url",
            "scrape_datetime",
            "contract_section",
        ]
    ].copy()

    # -----------------------------------------------------------------
    # BigQuery-friendly pandas dtypes
    # -----------------------------------------------------------------

    integer_columns = [
        "year",
        "cap_hit",
        "aav",
        "total_salary",
        "signing_bonus",
        "performance_bonus_amount",
    ]

    for col in integer_columns:
        df[col] = df[col].astype(
            "Int64"
        )

    boolean_columns = [
        "no_movement_clause",
        "no_trade_clause",
        "modified_no_trade_clause",
        "two_way_contract",
        "performance_bonus",
    ]

    for col in boolean_columns:
        df[col] = df[col].astype(
            "boolean"
        )

    # -----------------------------------------------------------------
    # QA
    # -----------------------------------------------------------------

    missing_position = (
        df["position"]
        .isna()
        .sum()
    )

    if missing_position:
        print(
            f"  WARNING: "
            f"{missing_position} contract-year rows "
            "have missing position."
        )

    duplicate_key = [
        "team_slug",
        "player_url",
        "season",
        "contract_section",
    ]

    duplicated = df.duplicated(
        subset=duplicate_key,
        keep=False,
    )

    if duplicated.any():

        print(
            "\nDuplicate player-season rows:"
        )

        print(
            df.loc[
                duplicated,
                duplicate_key,
            ]
            .sort_values(
                duplicate_key
            )
            .to_string(
                index=False
            )
        )

        raise ValueError(
            f"Duplicate player-season records "
            f"found for {team_name}."
        )

    return df


In [5]:
async def download():

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 "
                "(Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/139.0.0.0 "
                "Safari/537.36"
            )
        )

        page = await context.new_page()

        html = await fetch_team_html(
            page=page,
            team_name=TEAM["team_name"],
            url=TEAM["url"],
        )

        await browser.close()

        return html

html = await download()

print(len(html))

Loading New York Rangers...
  https://puckpedia.com/team/new-york-rangers
  HTTP status: 200
1236454


In [6]:
df = parse_contract_page(
    html=html,
    team_slug=TEAM["team_slug"],
    team_name=TEAM["team_name"],
    source_url=TEAM["url"],
)

print(df.shape)

df

Section: Forwards
Section: Defence
Section: Goaltenders
Section: Buried
Section: Non-roster Forwards
Section: Non-roster Defence
Section: Non-roster Goaltenders
(95, 21)


,team_slug,team_name,player,player_url,position,catches,year,season,cap_hit,aav,...,signing_bonus,performance_bonus_amount,no_movement_clause,no_trade_clause,modified_no_trade_clause,two_way_contract,performance_bonus,source_url,scrape_datetime,contract_section
0,new-york-rangers,New York Rangers,Pavel Dorofeyev,https://puckpedia.com/player/pavel-dorofeyev,"LW,RW",None,1,2026-27,11000000,11000000,...,13000000,0,False,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Forwards
1,new-york-rangers,New York Rangers,Pavel Dorofeyev,https://puckpedia.com/player/pavel-dorofeyev,"LW,RW",None,2,2027-28,11000000,11000000,...,6000000,0,False,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Forwards
2,new-york-rangers,New York Rangers,Pavel Dorofeyev,https://puckpedia.com/player/pavel-dorofeyev,"LW,RW",None,3,2028-29,11000000,11000000,...,5000000,0,True,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Forwards
3,new-york-rangers,New York Rangers,Pavel Dorofeyev,https://puckpedia.com/player/pavel-dorofeyev,"LW,RW",None,4,2029-30,11000000,11000000,...,4000000,0,True,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Forwards
4,new-york-rangers,New York Rangers,Pavel Dorofeyev,https://puckpedia.com/player/pavel-dorofeyev,"LW,RW",None,5,2030-31,11000000,11000000,...,4000000,0,True,False,True,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Forwards
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,new-york-rangers,New York Rangers,Callum Tung,https://puckpedia.com/player/callum-tung,G,L,1,2026-27,972500,1075000,...,97500,102500,False,False,False,True,True,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Non-roster Goaltenders
91,new-york-rangers,New York Rangers,Callum Tung,https://puckpedia.com/player/callum-tung,G,L,2,2027-28,972500,1075000,...,97500,102500,False,False,False,True,True,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Non-roster Goaltenders
92,new-york-rangers,New York Rangers,Dylan Garand,https://puckpedia.com/player/dylan-garand,G,L,1,2026-27,875000,875000,...,0,0,False,False,False,True,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Non-roster Goaltenders
93,new-york-rangers,New York Rangers,Dylan Garand,https://puckpedia.com/player/dylan-garand,G,L,2,2027-28,875000,875000,...,0,0,False,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Non-roster Goaltenders


In [7]:
df[df["player"] == "Juuso Parssinen"].sort_values(
    ["season", "contract_section"]
)

,team_slug,team_name,player,player_url,position,catches,year,season,cap_hit,aav,...,signing_bonus,performance_bonus_amount,no_movement_clause,no_trade_clause,modified_no_trade_clause,two_way_contract,performance_bonus,source_url,scrape_datetime,contract_section
52,new-york-rangers,New York Rangers,Juuso Parssinen,https://puckpedia.com/player/juuso-parssinen,"C,LW",None,1,2026-27,25000,1250000,...,0,0,False,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Buried
53,new-york-rangers,New York Rangers,Juuso Parssinen,https://puckpedia.com/player/juuso-parssinen,"C,LW",None,1,2026-27,1250000,1250000,...,0,0,False,False,False,False,False,https://puckpedia.com/team/new-york-rangers,2026-08-07 20:48:51.975077+00:00,Non-roster Forwards
